In [12]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import geopandas as gpd
import pygris

print("=" * 70)
print("Cleaned ACS + Tract Join (with proper data types)")
print("=" * 70)

# ============================================================
# 1. LOAD CENSUS TRACT GEOMETRIES
# ============================================================

ia_tracts = pygris.tracts(state="IA", year=2024, cb=True)
ok_tracts = pygris.tracts(state="OK", year=2024, cb=True)
tx_tracts = pygris.tracts(state="TX", year=2024, cb=True)

tracts = pd.concat([ia_tracts, ok_tracts, tx_tracts], ignore_index=True)
print(f"Total census tracts loaded: {len(tracts):,}")

# ============================================================
# 2. LOAD ACS CSV FILES + CLEAN DATA TYPES
# ============================================================

# Helper function to safely convert columns to numeric
def clean_numeric(df, col):
    """Convert column to numeric, coercing errors to NaN"""
    df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Load files (update paths if your filenames are different)
acs_pop     = pd.read_csv("data/external/ACSDT5Y2024.B01003.csv")
acs_housing = pd.read_csv("data/external/ACSDT5Y2024.B25001.csv")
acs_income  = pd.read_csv("data/external/ACSDT5Y2024.B19013.csv")
acs_value   = pd.read_csv("data/external/ACSDT5Y2024.B25077.csv")

# Clean and rename columns
acs_pop = clean_numeric(acs_pop, "B01003_001E")
acs_pop = acs_pop.rename(columns={"B01003_001E": "total_population"})

acs_housing = clean_numeric(acs_housing, "B25001_001E")
acs_housing = acs_housing.rename(columns={"B25001_001E": "total_housing_units"})

acs_income = clean_numeric(acs_income, "B19013_001E")
acs_income = acs_income.rename(columns={"B19013_001E": "median_household_income"})

acs_value = clean_numeric(acs_value, "B25077_001E")
acs_value = acs_value.rename(columns={"B25077_001E": "median_home_value"})

print("ACS files loaded and cleaned.")

# ============================================================
# 3. MERGE ALL ACS VARIABLES
# ============================================================

acs_combined = acs_pop[["GEO_ID", "total_population"]] \
    .merge(acs_housing[["GEO_ID", "total_housing_units"]], on="GEO_ID", how="left") \
    .merge(acs_income[["GEO_ID", "median_household_income"]], on="GEO_ID", how="left") \
    .merge(acs_value[["GEO_ID", "median_home_value"]], on="GEO_ID", how="left")

print(f"Combined ACS data shape: {acs_combined.shape}")

# ============================================================
# 4. JOIN ACS DATA WITH TRACT GEOMETRIES
# ============================================================

# Extract tract GEOID from ACS GEO_ID (last 11 characters)
acs_combined["GEOID"] = acs_combined["GEO_ID"].str[-11:]

# Merge with tract geometries
tracts_acs = tracts.merge(
    acs_combined[["GEOID", "total_population", "total_housing_units", 
                  "median_household_income", "median_home_value"]],
    on="GEOID",
    how="left"
)

print(f"Final joined dataset shape: {tracts_acs.shape}")

# ============================================================
# 5. SAVE CLEANED DATASET
# ============================================================

output_path = "data/processed/tracts_with_acs_2024.parquet"
tracts_acs.to_parquet(output_path, index=False)

print(f"✅ Cleaned file saved to: {output_path}")
print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Cleaned ACS + Tract Join (with proper data types)
Using FIPS code '19' for input 'IA'
Using FIPS code '40' for input 'OK'
Using FIPS code '48' for input 'TX'
Total census tracts loaded: 8,985
ACS files loaded and cleaned.
Combined ACS data shape: (8998, 5)
Final joined dataset shape: (8985, 18)
✅ Cleaned file saved to: data/processed/tracts_with_acs_2024.parquet


In [14]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

file_path = "data/processed/tracts_with_acs_2024.parquet"

print("=" * 70)
print("Inspecting Cleaned ACS + Tract Dataset")
print("=" * 70)

# ============================================================
# 1. BASIC INFORMATION
# ============================================================

print("\n--- 1. Basic Information ---")
con.sql(f"""
    SELECT 
        COUNT(*) AS total_tracts,
        COUNT(*) FILTER (WHERE total_population IS NOT NULL) AS tracts_with_population
    FROM '{file_path}'
""").show()

# ============================================================
# 2. COLUMN NAMES AND DATA TYPES
# ============================================================

print("\n--- 2. Column Names and Data Types ---")
con.sql(f"""
    DESCRIBE '{file_path}'
""").show()

# ============================================================
# 3. SUMMARY STATISTICS
# ============================================================

print("\n--- 3. Summary Statistics ---")
con.sql(f"""
    SELECT 
        MIN(total_population) AS min_pop,
        MAX(total_population) AS max_pop,
        ROUND(AVG(total_population), 0) AS avg_pop,

        MIN(total_housing_units) AS min_housing,
        MAX(total_housing_units) AS max_housing,
        ROUND(AVG(total_housing_units), 0) AS avg_housing,

        MIN(median_household_income) AS min_income,
        MAX(median_household_income) AS max_income,
        ROUND(AVG(median_household_income), 0) AS avg_income,

        MIN(median_home_value) AS min_home_value,
        MAX(median_home_value) AS max_home_value,
        ROUND(AVG(median_home_value), 0) AS avg_home_value
    FROM '{file_path}'
""").show()

# ============================================================
# 4. NULL VALUE CHECK
# ============================================================

print("\n--- 4. Null Value Count ---")
con.sql(f"""
    SELECT 
        COUNT(*) - COUNT(total_population) AS null_population,
        COUNT(*) - COUNT(median_household_income) AS null_income,
        COUNT(*) - COUNT(median_home_value) AS null_home_value
    FROM '{file_path}'
""").show()

# ============================================================
# 5. SAMPLE DATA
# ============================================================

print("\n--- 5. Sample Rows ---")
con.sql(f"""
    SELECT 
        GEOID,
        NAME,
        total_population,
        total_housing_units,
        median_household_income,
        median_home_value
    FROM '{file_path}'
    LIMIT 8
""").show()

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Inspecting Cleaned ACS + Tract Dataset

--- 1. Basic Information ---
┌──────────────┬────────────────────────┐
│ total_tracts │ tracts_with_population │
│    int64     │         int64          │
├──────────────┼────────────────────────┤
│         8985 │                   8985 │
└──────────────┴────────────────────────┘


--- 2. Column Names and Data Types ---
┌─────────────────────────┬───────────────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │      column_type      │  null   │   key   │ default │  extra  │
│         varchar         │        varchar        │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼───────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ STATEFP                 │ VARCHAR               │ YES     │ NULL    │ NULL    │ NULL    │
│ COUNTYFP                │ VARCHAR             

In [1]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import geopandas as gpd

print("=" * 70)
print("Adding More ACS Variables to Combined Dataset")
print("=" * 70)

# ============================================================
# 1. LOAD EXISTING COMBINED DATASET
# ============================================================

combined = gpd.read_parquet("data/processed/tracts_acs_wind_2024.parquet")
print(f"Existing combined data loaded: {len(combined):,} tracts")

# ============================================================
# 2. LOAD AND CLEAN NEW ACS TABLES
# ============================================================

# --- Educational Attainment (Bachelor's or higher) ---
edu = pd.read_csv("data/external/ACSDT5Y2024.B15003.csv")
# Sum the columns for Bachelor's, Master's, Professional, Doctorate
edu["bachelors_or_higher"] = (
    edu["B15003_022E"] + edu["B15003_023E"] + 
    edu["B15003_024E"] + edu["B15003_025E"]
)
edu = edu[["GEO_ID", "bachelors_or_higher"]]

# --- Employment ---
emp = pd.read_csv("data/external/ACSDT5Y2024.B23025.csv")
emp = emp.rename(columns={"B23025_004E": "employed"})
emp = emp[["GEO_ID", "employed"]]

# --- Poverty (optional but useful) ---
pov = pd.read_csv("data/external/ACSDT5Y2024.B17001.csv")
pov = pov.rename(columns={"B17001_002E": "below_poverty"})
pov = pov[["GEO_ID", "below_poverty"]]

print("New ACS tables loaded and cleaned.")

# ============================================================
# 3. MERGE NEW VARIABLES
# ============================================================

combined = combined.merge(edu, left_on="GEOID", right_on="GEO_ID", how="left")
combined = combined.merge(emp, left_on="GEOID", right_on="GEO_ID", how="left")
combined = combined.merge(pov, left_on="GEOID", right_on="GEO_ID", how="left")

# Drop duplicate GEO_ID columns if they exist
combined = combined.drop(columns=[col for col in combined.columns if col.startswith("GEO_ID")], errors="ignore")

print(f"Updated dataset shape: {combined.shape}")

# ============================================================
# 4. SAVE UPDATED DATASET
# ============================================================

output_path = "data/processed/tracts_acs_wind_2022_expanded.parquet"
combined.to_parquet(output_path, index=False)

print(f"✅ Expanded dataset saved to: {output_path}")
print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Adding More ACS Variables to Combined Dataset
Existing combined data loaded: 8,985 tracts
New ACS tables loaded and cleaned.
Updated dataset shape: (8985, 24)
✅ Expanded dataset saved to: data/processed/tracts_acs_wind_2022_expanded.parquet


/var/folders/1l/32pjxxc168qc488f3mcj2_2r0000gn/T/ipykernel_28984/1379480188.py:51: DtypeWarning: Columns (0: B17001_001E, 1: B17001_001M, 2: B17001_002E, 3: B17001_002M, 4: B17001_003E, 5: B17001_003M, 6: B17001_004E, 7: B17001_004M, 8: B17001_005E, 9: B17001_005M, 10: B17001_006E, 11: B17001_006M, 12: B17001_007E, 13: B17001_007M, 14: B17001_008E, 15: B17001_008M, 16: B17001_009E, 17: B17001_009M, 18: B17001_010E, 19: B17001_010M, 20: B17001_011E, 21: B17001_011M, 22: B17001_012E, 23: B17001_012M, 24: B17001_013E, 25: B17001_013M, 26: B17001_014E, 27: B17001_014M, 28: B17001_015E, 29: B17001_015M, 30: B17001_016E, 31: B17001_016M, 32: B17001_017E, 33: B17001_017M, 34: B17001_018E, 35: B17001_018M, 36: B17001_019E, 37: B17001_019M, 38: B17001_020E, 39: B17001_020M, 40: B17001_021E, 41: B17001_021M, 42: B17001_022E, 43: B17001_022M, 44: B17001_023E, 45: B17001_023M, 46: B17001_024E, 47: B17001_024M, 48: B17001_025E, 49: B17001_025M, 50: B17001_026E, 51: B17001_026M, 52: B17001_027E, 53:

In [2]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import geopandas as gpd

print("=" * 70)
print("Adding Year Structure Built Variable")
print("=" * 70)

# ============================================================
# 1. LOAD EXISTING COMBINED DATASET
# ============================================================

combined = gpd.read_parquet("data/processed/tracts_acs_wind_2024.parquet")
print(f"Existing combined data loaded: {len(combined):,} tracts")

# ============================================================
# 2. LOAD AND CLEAN YEAR STRUCTURE BUILT DATA (B25034)
# ============================================================

built = pd.read_csv("data/external/ACSDT5Y2024.B25034.csv")

# Create useful aggregated columns
# Built 2010 or later (recent development)
built["housing_built_2010_or_later"] = (
    built.get("B25034_002E", 0) + built.get("B25034_003E", 0)
)

# Built 2000–2009
built["housing_built_2000_to_2009"] = built.get("B25034_004E", 0)

# Built before 2000 (older housing stock)
built["housing_built_before_2000"] = (
    built.get("B25034_005E", 0) + built.get("B25034_006E", 0) +
    built.get("B25034_007E", 0) + built.get("B25034_008E", 0) +
    built.get("B25034_009E", 0) + built.get("B25034_010E", 0) +
    built.get("B25034_011E", 0)
)

# Keep only GEO_ID and the new columns
built = built[["GEO_ID", "housing_built_2010_or_later", 
               "housing_built_2000_to_2009", "housing_built_before_2000"]]

print("Year Structure Built data loaded and aggregated.")

# ============================================================
# 3. MERGE WITH EXISTING DATASET
# ============================================================

combined = combined.merge(
    built, 
    left_on="GEOID", 
    right_on="GEO_ID", 
    how="left"
)

# Drop duplicate GEO_ID column if it exists
if "GEO_ID" in combined.columns:
    combined = combined.drop(columns=["GEO_ID"])

print(f"Updated dataset shape: {combined.shape}")

# ============================================================
# 4. SAVE UPDATED DATASET
# ============================================================

output_path = "data/processed/tracts_acs_wind_2024_expanded.parquet"
combined.to_parquet(output_path, index=False)

print(f"✅ Expanded dataset saved to: {output_path}")
print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Adding Year Structure Built Variable
Existing combined data loaded: 8,985 tracts
Year Structure Built data loaded and aggregated.
Updated dataset shape: (8985, 24)
✅ Expanded dataset saved to: data/processed/tracts_acs_wind_2024_expanded.parquet


In [8]:
import pandas as pd

built = pd.read_csv("data/external/ACSDT5Y2024.B25034.csv")
print("Column names in your B25034 file:")
print(built.columns.tolist()[:15])   # Show first 15 columns

Column names in your B25034 file:
['GEO_ID', 'NAME', 'B25034_001E', 'B25034_001M', 'B25034_002E', 'B25034_002M', 'B25034_003E', 'B25034_003M', 'B25034_004E', 'B25034_004M', 'B25034_005E', 'B25034_005M', 'B25034_006E', 'B25034_006M', 'B25034_007E']


In [14]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import geopandas as gpd

print("=" * 70)
print("Adding Year Structure Built (Final Fixed Version)")
print("=" * 70)

# ============================================================
# 1. LOAD EXISTING COMBINED DATASET
# ============================================================

combined = gpd.read_parquet("data/processed/tracts_acs_wind_2024.parquet")
print(f"Existing data loaded: {len(combined):,} tracts")

# ============================================================
# 2. LOAD AND CLEAN YEAR STRUCTURE BUILT CSV
# ============================================================

built = pd.read_csv("data/external/ACSDT5Y2024.B25034.csv")

# Convert relevant columns to numeric
cols_to_convert = ['B25034_002E', 'B25034_003E', 'B25034_004E', 
                   'B25034_005E', 'B25034_006E', 'B25034_007E',
                   'B25034_008E', 'B25034_009E', 'B25034_010E', 'B25034_011E']

for col in cols_to_convert:
    if col in built.columns:
        built[col] = pd.to_numeric(built[col], errors='coerce').fillna(0)

# Create clean aggregated columns
built["housing_built_2010_or_later"] = built["B25034_002E"] + built["B25034_003E"]
built["housing_built_2000_to_2009"]  = built["B25034_004E"]
built["housing_built_before_2000"] = (
    built["B25034_005E"] + built["B25034_006E"] + built["B25034_007E"] +
    built["B25034_008E"] + built["B25034_009E"] + built["B25034_010E"] + built["B25034_011E"]
)

# Extract the 11-digit GEOID (last 11 characters of GEO_ID)
built["GEOID"] = built["GEO_ID"].str[-11:]

# Keep only what we need
built_clean = built[["GEOID", "housing_built_2010_or_later", 
                     "housing_built_2000_to_2009", "housing_built_before_2000"]].copy()

print("Year Structure Built data cleaned and GEOID extracted.")

# ============================================================
# 3. MERGE WITH EXISTING DATASET (using correct GEOID)
# ============================================================

combined = combined.merge(
    built_clean, 
    on="GEOID", 
    how="left"
)

print(f"Updated dataset shape: {combined.shape}")

# ============================================================
# 4. SAVE FINAL DATASET
# ============================================================

output_path = "data/processed/tracts_acs_wind_2024_expanded.parquet"
combined.to_parquet(output_path, index=False)

print(f"✅ Final expanded dataset saved to: {output_path}")
print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Adding Year Structure Built (Final Fixed Version)
Existing data loaded: 8,985 tracts
Year Structure Built data cleaned and GEOID extracted.
Updated dataset shape: (8985, 24)
✅ Final expanded dataset saved to: data/processed/tracts_acs_wind_2024_expanded.parquet


In [15]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

file_path = "data/processed/tracts_acs_wind_2024_expanded.parquet"

print("=" * 70)
print("Final Inspection: Expanded Dataset with Year Structure Built")
print("=" * 70)

# ============================================================
# 1. VERIFY NEW COLUMNS EXIST AND HAVE DATA
# ============================================================

print("\n--- 1. Column Names & Sample of New Housing Columns ---")
con.sql(f"""
    SELECT 
        column_name, 
        data_type
    FROM duckdb_columns()
    WHERE table_name LIKE '%tracts_acs_wind_2022_expanded%'
      AND column_name LIKE '%housing_built%'
""").show()

# ============================================================
# 2. SUMMARY STATISTICS (Should now show real numbers)
# ============================================================

print("\n--- 2. Housing Age Summary Statistics ---")
con.sql(f"""
    SELECT 
        ROUND(AVG(housing_built_2010_or_later), 0) AS avg_built_2010_later,
        ROUND(AVG(housing_built_2000_to_2009), 0) AS avg_built_2000_2009,
        ROUND(AVG(housing_built_before_2000), 0) AS avg_built_before_2000
    FROM '{file_path}'
""").show()

# ============================================================
# 3. COMPARE HOUSING AGE: With vs Without Wind Farms
# ============================================================

print("\n--- 3. Housing Age by Wind Farm Presence ---")
con.sql(f"""
    SELECT 
        has_wind_farm,
        COUNT(*) AS tract_count,
        ROUND(AVG(housing_built_2010_or_later), 0) AS avg_recent_housing,
        ROUND(AVG(housing_built_before_2000), 0) AS avg_old_housing
    FROM '{file_path}'
    GROUP BY has_wind_farm
    ORDER BY has_wind_farm
""").show()

# ============================================================
# 4. NULL CHECK (Should be much lower now)
# ============================================================

print("\n--- 4. Null Count Check ---")
con.sql(f"""
    SELECT 
        COUNT(*) - COUNT(housing_built_2010_or_later) AS null_built_2010_later,
        COUNT(*) - COUNT(housing_built_2000_to_2009) AS null_built_2000_2009,
        COUNT(*) - COUNT(housing_built_before_2000) AS null_built_before_2000
    FROM '{file_path}'
""").show()

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Final Inspection: Expanded Dataset with Year Structure Built

--- 1. Column Names & Sample of New Housing Columns ---
┌─────────────┬───────────┐
│ column_name │ data_type │
│   varchar   │  varchar  │
└─────────────┴───────────┘
          0 rows         


--- 2. Housing Age Summary Statistics ---
┌──────────────────────┬─────────────────────┬───────────────────────┐
│ avg_built_2010_later │ avg_built_2000_2009 │ avg_built_before_2000 │
│        double        │       double        │        double         │
├──────────────────────┼─────────────────────┼───────────────────────┤
│                333.0 │               285.0 │                1090.0 │
└──────────────────────┴─────────────────────┴───────────────────────┘


--- 3. Housing Age by Wind Farm Presence ---
┌───────────────┬─────────────┬────────────────────┬─────────────────┐
│ has_wind_farm │ tract_co